In [21]:
import os
import scanpy as sc
import pandas as pd
import anndata as ad
import numpy as np
import matplotlib.pyplot as plt
import json

## Load Data

In [2]:
filtered_data_path = f"{os.environ['HOME']}/.cz-benchmarks/datasets/replogle_k562_essential_perturbpredict_de_results_control_cells.h5ad"
# filtered_data_path = "/data2/czbenchmarks/replogle2022/K562_essential_raw_singlecell_01.h5ad"
adata_filtered = ad.read_h5ad(filtered_data_path, backed=None)

In [4]:
de_results = pd.DataFrame(adata_filtered.uns[f"de_results_wilcoxon"])
# de_results['condition'] = de_results['condition'].cat.remove_unused_categories()
de_results['condition'] = de_results['condition'].astype(str)

control_cells_ids_input = adata_filtered.uns["control_cells_ids"]
control_cells_ids_input.pop("non-targeting", None)

array([], dtype=float64)

In [23]:
with open("../control_cells_ids_replogle_k562_essential_perturbpredict.json", "r") as f:
    control_cells_ids_new = json.load(f)

with open("../control_cells_ids_replogle_k562_essential_perturbpredict_validate.json", "r") as f:
    control_cells_ids_validate = json.load(f)

## QC of Conditions

In [24]:
len(control_cells_ids_input.keys()), len(control_cells_ids_new.keys()), len(control_cells_ids_validate.keys())

(2057, 2057, 2057)

In [25]:
set(control_cells_ids_new.keys()) == set(control_cells_ids_input.keys()) == set(control_cells_ids_validate.keys())

True

In [26]:
set(control_cells_ids_new.keys()).issubset(set(control_cells_ids_input.keys()))

True

In [27]:
set(control_cells_ids_validate.keys()).issubset(set(control_cells_ids_input.keys()))

True

## QC of Matches

In [28]:
conditions = list(control_cells_ids_new.keys())

In [38]:
for condition in conditions:
    print(condition, len(control_cells_ids_input[condition]), len(control_cells_ids_validate[condition]))
    assert len(control_cells_ids_input[condition]) == len(control_cells_ids_validate[condition]) == len(control_cells_ids_new[condition])
    # assert list(control_cells_ids_input[condition]) == list(control_cells_ids_validate[condition].values())
    matches = 0
    for pos, (val1, val2) in enumerate(zip(control_cells_ids_input[condition], control_cells_ids_validate[condition].values())):
        if val1 == val2:
            matches += 1
    print(f"     matches {matches}, differences {len(control_cells_ids_input[condition]) - matches}")

ENSG00000145414 94 94
     matches 93, differences 1
ENSG00000169679 648 648
     matches 644, differences 4
ENSG00000198258 232 232
     matches 230, differences 2
ENSG00000171159 212 212
     matches 209, differences 3
ENSG00000100575 201 201
     matches 200, differences 1
ENSG00000198952 83 83
     matches 83, differences 0
ENSG00000143373 205 205
     matches 204, differences 1
ENSG00000141026 119 119
     matches 119, differences 0
ENSG00000100316 1996 1996
     matches 1979, differences 17
ENSG00000179988 379 379
     matches 377, differences 2
ENSG00000112130 176 176
     matches 174, differences 2
ENSG00000125686 254 254
     matches 254, differences 0
ENSG00000169045 160 160
     matches 160, differences 0
ENSG00000198755 257 257
     matches 255, differences 2
ENSG00000121073 70 70
     matches 68, differences 2
ENSG00000189308 306 306
     matches 305, differences 1
ENSG00000173674 134 134
     matches 133, differences 1
ENSG00000185385 108 108
     matches 108, differences

## Are all cells used

In [42]:
control_cells_used = set()
for condition in conditions:
    control_cells_used = control_cells_used.union(set(control_cells_ids_new[condition].keys()))
    control_cells_used = control_cells_used.union(set(control_cells_ids_new[condition].values()))
    

In [43]:
all_cells = set(adata_filtered.obs_names)

In [44]:
len(all_cells), len(control_cells_used)

(310385, 310281)

In [45]:
control_cells_used.issubset(all_cells)

True

In [50]:
unused_cell_ids = all_cells.difference(control_cells_used)

In [51]:
unused_cells = adata_filtered.obs.loc[list(unused_cell_ids)]

In [54]:
unused_cells.condition.astype(str).unique()

array(['non-targeting'], dtype=object)

In [52]:
unused_cells

,gem_group,condition_name,condition,transcript,gene_transcript,sgID_AB,mitopercent,UMI_count,z_gemgroup_UMI,core_scale_factor,core_adjusted_UMI_count
cell_barcode,,,,,,,,,,,
GGGAAGTCAGCAGTTT-41,41,non-targeting,non-targeting,non-targeting,11267_non-targeting_non-targeting_non-targeting,non-targeting_03441|non-targeting_00663,0.099225,10965.0,-0.421027,0.943084,11626.750977
CAGTTAGAGATCACCT-30,30,non-targeting,non-targeting,non-targeting,11176_non-targeting_non-targeting_non-targeting,non-targeting_02774|non-targeting_00697,0.099290,14090.0,0.204779,0.949840,14834.083984
CGAAGGAGTACGAGCA-36,36,non-targeting,non-targeting,non-targeting,10866_non-targeting_non-targeting_non-targeting,non-targeting_00678|non-targeting_02387,0.092739,18881.0,0.978741,1.011235,18671.232422
CTGCATCAGTTACGTC-26,26,non-targeting,non-targeting,non-targeting,11290_non-targeting_non-targeting_non-targeting,non-targeting_03530|non-targeting_01407,0.101344,9897.0,-0.664785,0.901381,10979.821289
CTTCTCTAGGGTTGCA-20,20,non-targeting,non-targeting,non-targeting,10841_non-targeting_non-targeting_non-targeting,non-targeting_00531|non-targeting_01795,0.083502,16838.0,0.557413,1.030337,16342.218750
...,...,...,...,...,...,...,...,...,...,...,...
AGTGTTGCAGCCCACA-14,14,non-targeting,non-targeting,non-targeting,11305_non-targeting_non-targeting_non-targeting,non-targeting_03624|non-targeting_03589,0.127325,14137.0,-0.129345,1.050055,13463.107422
CACAACATCTTCACGC-9,9,non-targeting,non-targeting,non-targeting,10759_non-targeting_non-targeting_non-targeting,non-targeting_00081|non-targeting_03672,0.048635,17107.0,0.507703,1.036454,16505.314453
TAATTCCCATTCACAG-11,11,non-targeting,non-targeting,non-targeting,11251_non-targeting_non-targeting_non-targeting,non-targeting_03290|non-targeting_03021,0.100437,17842.0,0.629333,1.050188,16989.341797
